## Planning:
- Load data
- Maximum Parameter with Optuna
- Blending with Optuna
- Output

In [1]:
import numpy as np
import pandas as pd
import optuna
from sklearn.metrics import roc_auc_score

# import OS
import os

In [2]:
from pathlib import Path

In [3]:
current_dir = Path.cwd()
competition_root = current_dir.parents[1]

In [4]:
competition_path = competition_root/"Predicting Credit Card Fraud"

In [5]:
train = pd.read_csv(competition_path/"data/train.csv")
train_data = train[["id","is_fraud"]]

test = pd.read_csv(competition_path/"data/test.csv")
test_data = test[["id"]]

submit = pd.read_csv(competition_path/"data/sample_submission.csv")

In [6]:
folder_oof = competition_path/"oof"
folder_sub = competition_path/"sub"

In [7]:
content_oof = os.listdir(folder_oof)
content_sub = os.listdir(folder_sub)

In [8]:
for oof in content_oof:
    path_oof = folder_oof/oof
    oof_data = pd.read_csv(path_oof)
    train_data = pd.concat([train_data,oof_data.iloc[:,[1]]],axis=1)

In [9]:
for sub in content_sub:
    path_sub = folder_sub/sub
    sub_data = pd.read_csv(path_sub)
    test_data = pd.concat([test_data,sub_data.iloc[:,[1]]],axis=1)

## Data Process for Modeling

In [10]:
X = train_data.drop(columns=["id","is_fraud"])
y = train_data["is_fraud"]
X_test = test_data.drop(columns=["id"])
X_test.columns = X.columns

In [11]:
X = X.drop(columns=["log_5cv"])
X_test = X_test.drop(columns=["log_5cv"])

In [12]:
oof_matrix = X.values 
y_true_array = y.values
num_models = oof_matrix.shape[1]

In [13]:
def objective_optuna(trial):
  weights = [trial.suggest_float(f"w_{i}", 0.0, 1.0) for i in range(num_models)]
  sum_w = sum(weights)
  if sum_w == 0:
    norm_w = weights
  else:
    norm_w = [w / sum_w for w in weights]

  blend = np.dot(X.values, norm_w)
  return roc_auc_score(y, blend)  # Optuna maximizes by default

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective_optuna, n_trials=1000)

[I 2026-11-09 19:35:28,628] A new study created in memory with name: no-name-e54fe356-9740-4393-b4e4-a97b5139c62d
[I 2026-11-09 19:35:28,979] Trial 0 finished with value: 0.9449417228320767 and parameters: {'w_0': 0.7747338224307293, 'w_1': 0.492499734694823, 'w_2': 0.01879431179498614, 'w_3': 0.8232932376519612, 'w_4': 0.5804892531395481, 'w_5': 0.9254111689061957, 'w_6': 0.9114865369459959, 'w_7': 0.9553644100065481, 'w_8': 0.6663148349063804, 'w_9': 0.13317498194546118, 'w_10': 0.21934726380424718, 'w_11': 0.9358182587560142, 'w_12': 0.7341207838511933, 'w_13': 0.5174206601917476, 'w_14': 0.776512206607729, 'w_15': 0.3190648265070487, 'w_16': 0.390410719298816, 'w_17': 0.30959262373037755, 'w_18': 0.7996383343428521, 'w_19': 0.7672642830251464, 'w_20': 0.06314573881690644, 'w_21': 0.4315959138684773, 'w_22': 0.8433054783961103, 'w_23': 0.061960989166872316, 'w_24': 0.7142020828615745, 'w_25': 0.5821847584348698, 'w_26': 0.8268580326591798, 'w_27': 0.9774671069032067, 'w_28': 0.89384

In [63]:
# 1. Get the best trial from your study
best_trial = study.best_trial

# 2. Extract the weights corresponding to the best trial's parameters
best_params = best_trial.params
raw_best_weights = np.array([best_params[f"w_{i}"] for i in range(num_models)])

# 3. Normalize the weights so they sum to 1 (matching your objective function logic)
sum_w = np.sum(raw_best_weights)
if sum_w == 0:
  final_weights = raw_best_weights
else:
  final_weights = raw_best_weights / sum_w

# 4. Print the best validation score and weights
print(f"Best Optuna ROC-AUC Score: {best_trial.value:.4f}")
print(f"Optimized Weights: {np.round(final_weights, 4)}")

# 5. Compute the final blended predictions on the test set
# X_test contains your test OOF columns (excluding 'id')
final_blend_test = np.dot(X_test.values, final_weights)

# 6. Assign to your submission dataframe and save
submit["is_fraud"] = final_blend_test
submit.to_csv(competition_path / "Blending Optuna v1.csv", index=False)
print("Submission file successfully saved!")

🥇 Best OOF AUC Score: 0.947149


In [64]:
final_weights

array([0.02513207, 0.0020922 , 0.07526872, 0.33463291, 0.21700368,
       0.0145255 , 0.10536397, 0.03705993, 0.02664652, 0.02387045,
       0.02105168, 0.02136036, 0.01550031, 0.01187116, 0.00695022,
       0.0076834 , 0.00757164, 0.00769768, 0.03871762])

In [66]:
# Compute the final blended predictions
final_blend = np.dot(X_test.values, final_weights)

In [67]:
submit["is_fraud"] = final_blend

In [68]:
submit.to_csv("Blending Optuna v2.csv",index=False)